# Fake News Detector for Nigerian Online Media (English + Pidgin)

This notebook builds the whole project described in Chapter Three:

1. Creates the project files (data collection, preprocessing, training, Streamlit app, tests)
2. Installs the libraries
3. Collects fact-checked claims from Dubawa and Africa Check (Google Fact Check Tools API + archive scraping)
4. Preprocesses: label harmonisation, language identification, duplicate/conflict removal, cleaning, 80/10/10 split
5. Trains Naive Bayes, Logistic Regression and Random Forest (TF-IDF, k-fold CV, SMOTE on training data only)
6. Shows the Chapter Four results: metrics, AUC-ROC, Brier score, confusion matrices, bias-audit tables
7. Runs the tests, previews the app, and downloads a deploy-ready zip for GitHub + Streamlit Community Cloud

**Run the cells from top to bottom.** A normal CPU runtime is enough, no GPU needed.

## Step 1 - Project folder

In [ ]:
import os

# Set to True to keep everything in Google Drive (survives Colab disconnects)
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/fake-news-detector-ng"
else:
    PROJECT = "/content/fake-news-detector-ng"

for d in ["data/raw", "models", "tests", ".streamlit"]:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
%cd {PROJECT}
print("Project folder:", PROJECT)

## Step 2 - Write the project files
Each cell below writes one file. These are the same files that get deployed.

In [ ]:
%%writefile requirements.txt
# Exact scikit-learn / imbalanced-learn versions so models trained in Colab
# load identically on Streamlit Community Cloud. Do not change one without the other.
scikit-learn==1.6.1
imbalanced-learn==0.13.0
joblib>=1.3,<2
numpy>=1.26,<2.3
pandas>=2.1,<2.4
scipy>=1.11,<1.16
nltk>=3.8,<4
lime==0.2.0.1
streamlit>=1.40,<2
requests>=2.31,<3
beautifulsoup4>=4.12,<5
matplotlib>=3.7,<4
seaborn>=0.13,<0.14
pytest>=7

In [ ]:
%%writefile preprocess.py
"""
preprocess.py
=============
Stage 1-2 of the pipeline (Chapter Three, Sections 3.4.2.1 - 3.4.2.4).

Takes one or more raw claim CSVs (sample, Google Fact Check API, Dubawa and
Africa Check scrapes, or a hand-collected file) and produces a clean,
labelled, deduplicated dataset split 80/10/10 into train / val / test.

Steps, in the order the chapter describes them:
  1. Label harmonisation   True -> Real ; False / Misleading / Mostly False -> Fake ;
                           Unverifiable / Unproven / unclear -> dropped (3.4.2.3)
  2. Language identification pass (English vs Nigerian Pidgin / code-switched)
  3. Topical category + group tags used for bias auditing (3.4.5, 3.4.7):
     category, region (geopolitical zone), party_ref, religion_ref, ethnic_ref
  4. Duplicate removal: exact (hash of normalised text) and near-duplicate
     (TF-IDF cosine > 0.90 within the same category); conflicting verdicts
     for the same claim are excluded entirely (3.4.2.3)
  5. Cleaning: HTML, URLs, emojis, personal data (emails, phone numbers,
     @handles) removed; Nigeria-adapted stopwords; stemming applied to
     English records only (3.4.2.4)
  6. 80/10/10 split stratified on label (+ source where possible).
     SMOTE is NOT applied here - it is applied to the training partition
     only, inside train_model.py, so nothing synthetic leaks into val/test.

Usage:
  python preprocess.py                         # all CSVs in data/raw/
  python preprocess.py --inputs data/raw/api_claims.csv data/raw/dubawa_claims.csv
  python preprocess.py --include-sample        # keep the sample file even if real data exists
"""

import argparse
import glob
import hashlib
import html
import json
import os
import re
from datetime import datetime, timezone
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
NEAR_DUP_THRESHOLD = 0.90
SAMPLE_FILENAME = "sample_claims.csv"

# ---------------------------------------------------------------------------
# 1. Label harmonisation
# ---------------------------------------------------------------------------
# Edit this table if your sources use other rating words. None = excluded.
VERDICT_MAP = {
    # Real
    "true": "Real", "correct": "Real", "mostly true": "Real",
    "mostly correct": "Real", "verified": "Real", "accurate": "Real",
    "real": "Real",
    # Fake (Misleading maps to Fake - justified in Section 3.4.2.3)
    "false": "Fake", "fake": "Fake", "misleading": "Fake",
    "mostly false": "Fake", "incorrect": "Fake", "exaggerated": "Fake",
    "fabricated": "Fake", "manipulated": "Fake", "altered": "Fake",
    "doctored": "Fake", "scam": "Fake", "hoax": "Fake",
    "partly false": "Fake", "partially false": "Fake",
    "false context": "Fake", "misleading context": "Fake",
    # Excluded - no defensible direction
    "unverifiable": None, "unverified": None, "unproven": None,
    "unclear": None, "insufficient evidence": None, "investigating": None,
    "satire": None, "explainer": None, "checked": None, "understated": None,
    "half true": None, "partly true": None, "mixed": None, "no evidence": None,
    "unrated": None,
}
_VERDICT_KEYS = sorted(VERDICT_MAP, key=len, reverse=True)


def _norm_verdict(verdict) -> str:
    v = str(verdict if verdict is not None else "").strip().lower()
    v = re.sub(r"[\s_\-]+", " ", v)
    return v.strip(" !.:;,\"'")


def harmonize_label(verdict):
    """Map a fact-checker rating (a word or a full sentence) to Real / Fake / None."""
    v = _norm_verdict(verdict)
    if not v or v == "nan":
        return None
    if v in VERDICT_MAP:
        return VERDICT_MAP[v]
    # Ratings often arrive as sentences: "False. The photo is from 2019..."
    for key in _VERDICT_KEYS:
        if re.match(rf"^{re.escape(key)}(?![a-z])", v):
            return VERDICT_MAP[key]
    return None


# ---------------------------------------------------------------------------
# 2. Language identification (English vs Nigerian Pidgin / code-switched)
# ---------------------------------------------------------------------------
PIDGIN_STRONG = {
    "dey", "wetin", "abeg", "una", "wahala", "sabi", "pikin", "comot", "naija",
    "wey", "dem", "sef", "oya", "haba", "don", "dis", "dat", "pipu", "pipo",
    "tok", "shey", "shebi", "abi", "wan", "katakata", "gbege", "yarn", "nna",
    "sey", "kpatakpata", "na", "unu", "ehn", "nawa",
}
PIDGIN_WEAK = {"oga", "sha", "chop", "waka", "fit", "okada", "kuku", "jare", "o"}
PIDGIN_PATTERNS = [
    r"\bfor (?:whole|di|dis|dat)\b",
    r"\bfit (?!the\b|a\b|an\b|in\b|into\b|for\b|with\b|and\b|to\b)[a-z]+",
    r"\bgo (?:dey|don)\b",
]


def detect_language(text) -> str:
    """Return 'pcm' (Pidgin / code-switched) or 'en' (standard Nigerian English)."""
    t = str(text).lower()
    tokens = re.findall(r"[a-z']+", t)
    score = sum(1.0 for w in tokens if w in PIDGIN_STRONG)
    score += sum(0.5 for w in tokens if w in PIDGIN_WEAK)
    score += sum(1.0 for p in PIDGIN_PATTERNS if re.search(p, t))
    return "pcm" if score >= 1.0 else "en"


# ---------------------------------------------------------------------------
# 3. Category and bias-audit group tags
# ---------------------------------------------------------------------------
CATEGORY_KEYWORDS = {
    "politics": [
        "inec", "election", "elections", "president", "presidency", "governor",
        "senate", "senator", "house of reps", "apc", "pdp", "labour party",
        "nnpp", "tinubu", "atiku", "peter obi", "obi", "buhari", "jonathan",
        "shettima", "vote", "votes", "voting", "bvas", "irev", "minister",
        "assembly", "campaign", "ballot", "polling", "lawmaker", "politician",
        "party", "tribunal", "candidate", "aso rock", "wike",
    ],
    "health": [
        "covid", "covid19", "coronavirus", "vaccine", "vaccines", "malaria",
        "cure", "cures", "disease", "hospital", "doctor", "doctors", "cancer",
        "virus", "health", "drug", "drugs", "ncdc", "world health organization", "lassa", "cholera",
        "monkeypox", "mpox", "diabetes", "hiv", "aids", "herbal", "medicine",
        "pregnant", "pregnancy", "polio", "nafdac", "infection", "treatment",
    ],
    "security": [
        "boko haram", "bandit", "bandits", "kidnap", "kidnapped", "kidnapping",
        "attack", "attacked", "army", "police", "killed", "killing", "terrorist",
        "terrorists", "ipob", "herdsmen", "military", "gunmen", "explosion",
        "bomb", "iswap", "soldiers", "insurgent", "insurgents", "dss", "efcc",
        "arrest", "arrested", "shooting", "unknown gunmen", "esn",
    ],
    "entertainment": [
        "bbnaija", "big brother", "nollywood", "actor", "actress", "singer",
        "music", "davido", "wizkid", "burna boy", "celebrity", "movie", "film",
        "football", "super eagles", "afcon", "osimhen", "tiktok star",
        "comedian", "album", "concert", "award", "musician", "rapper",
    ],
}
CATEGORY_PRIORITY = ["politics", "health", "security", "entertainment"]

ZONE_KEYWORDS = {
    "North-Central": ["benue", "kogi", "kwara", "nasarawa", "niger state",
                      "plateau state", "fct", "abuja", "jos", "ilorin",
                      "makurdi", "lokoja", "minna", "lafia"],
    "North-East": ["adamawa", "bauchi", "borno", "gombe", "taraba", "yobe",
                   "maiduguri", "yola", "jalingo", "damaturu"],
    "North-West": ["jigawa", "kaduna", "kano", "katsina", "kebbi", "sokoto",
                   "zamfara", "zaria", "dutse", "birnin kebbi", "gusau"],
    "South-East": ["abia", "anambra", "ebonyi", "enugu", "imo", "owerri",
                   "onitsha", "awka", "abakaliki", "umuahia", "nnewi"],
    "South-South": ["akwa ibom", "bayelsa", "cross river", "delta state",
                    "edo", "rivers state", "port harcourt", "uyo", "calabar",
                    "benin city", "warri", "asaba", "yenagoa", "niger delta",
                    "delta"],
    "South-West": ["ekiti", "lagos", "ogun", "ondo", "osun", "oyo", "ibadan",
                   "abeokuta", "akure", "osogbo", "ikeja", "lekki",
                   "ado ekiti"],
}
PARTY_KEYWORDS = {
    "APC": ["apc", "all progressives congress"],
    "PDP": ["pdp", "peoples democratic party"],
    "LP": ["labour party", "obidient", "obidients"],
    "NNPP": ["nnpp", "kwankwaso"],
    "APGA": ["apga"],
    "ADC": ["adc", "african democratic congress"],
}
RELIGION_KEYWORDS = {
    "Christian": ["christian", "christians", "church", "churches", "pastor",
                  "bishop", "catholic", "pentecostal", "priest"],
    "Muslim": ["muslim", "muslims", "islam", "islamic", "mosque", "imam",
               "sharia", "hijab", "jumat", "emir", "ramadan"],
}
ETHNIC_KEYWORDS = {
    "Igbo": ["igbo", "ndigbo", "biafra", "biafran"],
    "Yoruba": ["yoruba", "oduduwa"],
    "Hausa": ["hausa"],
    "Fulani": ["fulani"],
    "Ijaw": ["ijaw", "izon"],
    "Tiv": ["tiv"],
    "Kanuri": ["kanuri"],
    "Other": ["ogoni", "ibibio", "efik", "itsekiri", "urhobo", "isoko", "nupe",
              "idoma", "igala", "berom"],
}
GROUP_COLUMNS = ["language_tag", "category", "source", "region", "party_ref",
                 "religion_ref", "ethnic_ref"]


def _count_hits(text, keywords):
    return sum(1 for k in keywords if re.search(rf"\b{re.escape(k)}\b", text))


def _plain(text) -> str:
    t = html.unescape(str(text)).lower()
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return re.sub(r"\s+", " ", t).strip()


def tag_category(text) -> str:
    t = _plain(text)
    scores = {c: _count_hits(t, kws) for c, kws in CATEGORY_KEYWORDS.items()}
    best = max(scores.values())
    if best == 0:
        return "other"
    for c in CATEGORY_PRIORITY:
        if scores[c] == best:
            return c
    return "other"


def _multi_tag(text, table, none_value="None"):
    t = _plain(text)
    hits = [name for name, kws in table.items() if _count_hits(t, kws) > 0]
    if not hits:
        return none_value
    return hits[0] if len(hits) == 1 else "Multiple"


def tag_region(text) -> str:
    t = _plain(text)
    t = re.sub(r"\bniger republic\b", " ", t)
    hits = [z for z, kws in ZONE_KEYWORDS.items() if _count_hits(t, kws) > 0]
    if not hits:
        return "National/Unspecified"
    return hits[0] if len(hits) == 1 else "Multiple"


def tag_groups(text) -> dict:
    return {
        "category": tag_category(text),
        "region": tag_region(text),
        "party_ref": _multi_tag(text, PARTY_KEYWORDS),
        "religion_ref": _multi_tag(text, RELIGION_KEYWORDS),
        "ethnic_ref": _multi_tag(text, ETHNIC_KEYWORDS),
    }


# ---------------------------------------------------------------------------
# 5. Cleaning
# ---------------------------------------------------------------------------
ENGLISH_STOPWORDS = {
    "a", "about", "above", "after", "again", "against", "all", "am", "an",
    "and", "any", "are", "aren't", "as", "at", "be", "because", "been",
    "before", "being", "below", "between", "both", "but", "by", "can",
    "could", "did", "do", "does", "doing", "down", "during",
    "each", "few", "for", "from", "further", "had", "has", "have",
    "having", "he", "her", "here", "hers", "herself", "him", "himself",
    "his", "how", "i", "if", "in", "into", "is", "it", "its", "itself",
    "just", "me", "more", "most", "my", "myself", "nor",
    "now", "of", "off", "on", "once", "only", "or", "other", "our",
    "ours", "ourselves", "out", "over", "own", "same", "she", "should",
    "so", "some", "such", "than", "that", "the", "their", "theirs",
    "them", "themselves", "then", "there", "these", "they", "this",
    "those", "through", "to", "too", "under", "until", "up", "very",
    "was", "we", "were", "what", "when", "where", "which", "while",
    "who", "whom", "why", "will", "with", "you", "your", "yours",
    "yourself", "yourselves",
}
# NOTE: negations ("no", "not", "never", "don't") are deliberately kept -
# they flip the meaning of a claim.

# Nigerian Pidgin function words. Content words like "wahala", "sabi",
# "tok" are deliberately NOT included - they carry meaning.
PIDGIN_STOPWORDS = {
    "na", "dey", "dis", "dat", "dem", "una", "unu", "abeg", "sha", "sef",
    "wetin", "wey", "don", "fit", "wan", "shey", "abi", "nna", "biko",
    "chai", "e", "im", "pipu", "pipo",
}
STOPWORDS = ENGLISH_STOPWORDS | PIDGIN_STOPWORDS
# Pidgin content words are never stemmed even inside English records
PROTECTED_FROM_STEMMING = PIDGIN_STRONG | PIDGIN_WEAK | {"naira", "bvas", "inec"}

_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
_PHONE = re.compile(r"(?:\+?234|\b0)[789][01]\d[\s-]?\d{3}[\s-]?\d{4}\b")
_URL = re.compile(r"https?://\S+|www\.\S+")
_HANDLE = re.compile(r"@\w+")


def scrub_pii(text) -> str:
    """Remove personal data (emails, phone numbers, @handles). Used before logging."""
    t = str(text)
    t = _EMAIL.sub("[email]", t)
    t = _PHONE.sub("[phone]", t)
    t = _HANDLE.sub("[user]", t)
    return t


@lru_cache(maxsize=1)
def _stemmer():
    from nltk.stem import PorterStemmer  # no corpus download needed
    return PorterStemmer()


@lru_cache(maxsize=50000)
def _stem(word):
    return _stemmer().stem(word)


def clean_text(text, lang=None, stem=True) -> str:
    """Clean one claim. The same function is used at training time and in the app."""
    raw = html.unescape(str(text if text is not None else ""))
    if lang is None:
        lang = detect_language(raw)
    t = raw.lower()
    t = re.sub(r"<[^>]+>", " ", t)          # HTML tags
    t = _URL.sub(" ", t)                     # URLs
    t = _EMAIL.sub(" ", t)                   # emails (personal data)
    t = _PHONE.sub(" ", t)                   # phone numbers (personal data)
    t = _HANDLE.sub(" ", t)                  # @handles (personal data)
    t = re.sub(r"[^\w\s]", " ", t)           # punctuation AND emojis
    t = t.replace("_", " ")
    t = re.sub(r"\b\d+\b", " ", t)           # standalone numbers (keeps 5g, covid19)
    tokens = []
    for w in t.split():
        if w in STOPWORDS:
            continue
        if len(w) < 2 or (len(w) == 2 and not any(c.isdigit() for c in w)):
            continue
        if stem and lang == "en" and w not in PROTECTED_FROM_STEMMING:
            w = _stem(w)
        tokens.append(w)
    return " ".join(tokens)


def normalise_for_dedup(text) -> str:
    return _plain(text)


# ---------------------------------------------------------------------------
# 4. Duplicate and conflict removal
# ---------------------------------------------------------------------------
def _find(parent, i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def remove_duplicates(df: pd.DataFrame, log: dict) -> pd.DataFrame:
    df = df.copy()
    df["_norm"] = df["claim_text"].apply(normalise_for_dedup)
    df["_hash"] = df["_norm"].apply(lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest())

    # Pass 1: exact duplicates (same normalised text)
    keep_idx, exact_removed, conflict_removed = [], 0, 0
    conflicts = []
    for _, grp in df.groupby("_hash", sort=False):
        if grp["label"].nunique() > 1:
            conflict_removed += len(grp)
            conflicts.append(grp["claim_text"].iloc[0][:120])
            continue
        keep_idx.append(grp.index[0])
        exact_removed += len(grp) - 1
    df = df.loc[sorted(keep_idx)].copy()

    # Pass 2: near-duplicates within the same category (cosine > threshold)
    near_removed, near_conflict_removed = 0, 0
    keep_idx = []
    for _, grp in df.groupby("category", sort=False):
        idx = list(grp.index)
        if len(idx) < 2:
            keep_idx.extend(idx)
            continue
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform(grp["_norm"])
        sim = (X @ X.T).tocoo()
        parent = list(range(len(idx)))
        for i, j, s in zip(sim.row, sim.col, sim.data):
            if i < j and s > NEAR_DUP_THRESHOLD:
                ri, rj = _find(parent, i), _find(parent, j)
                if ri != rj:
                    parent[rj] = ri
        clusters = {}
        for k in range(len(idx)):
            clusters.setdefault(_find(parent, k), []).append(idx[k])
        for members in clusters.values():
            if len(members) == 1:
                keep_idx.append(members[0])
                continue
            if df.loc[members, "label"].nunique() > 1:
                near_conflict_removed += len(members)
                conflicts.append(df.loc[members[0], "claim_text"][:120])
                continue
            keep_idx.append(members[0])
            near_removed += len(members) - 1
    df = df.loc[sorted(keep_idx)].drop(columns=["_norm", "_hash"])

    log["exact_duplicates_removed"] = int(exact_removed)
    log["near_duplicates_removed"] = int(near_removed)
    log["conflicting_verdicts_removed"] = int(conflict_removed + near_conflict_removed)
    log["conflicting_claims_examples"] = conflicts[:20]
    return df


# ---------------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------------
OPTIONAL_COLS = ["claim_id", "source", "language_tag", "category", "date", "url"]


def resolve_inputs(inputs, raw_dir, include_sample):
    if inputs:
        paths = []
        for p in inputs:
            paths.extend(sorted(glob.glob(p)) or [p])
        return paths
    paths = sorted(glob.glob(os.path.join(raw_dir, "*.csv")))
    real = [p for p in paths if os.path.basename(p) != SAMPLE_FILENAME]
    if real and not include_sample:
        print(f"Real data found - excluding {SAMPLE_FILENAME}. Use --include-sample to keep it.")
        return real
    return paths


def load_raw(paths) -> pd.DataFrame:
    frames = []
    for p in paths:
        if not os.path.exists(p) or os.path.getsize(p) == 0:
            print(f"Skipping missing/empty file: {p}")
            continue
        df = pd.read_csv(p, dtype=str, keep_default_na=False)
        if os.path.basename(p).startswith("needs_review"):
            # Scraper review files: only rows a team member has filled in are used
            if "claim_text" not in df.columns and "claim_guess" in df.columns:
                df = df.rename(columns={"claim_guess": "claim_text"})
            df = df[df.get("verdict", pd.Series(dtype=str)).astype(str).str.strip() != ""]
            if df.empty:
                print(f"Skipping {p}: no verdicts filled in yet")
                continue
        if "verdict" not in df.columns and "label" in df.columns:
            df["verdict"] = df["label"]
        missing = {"claim_text", "verdict"} - set(df.columns)
        if missing:
            raise ValueError(f"{p} is missing required columns: {missing}")
        for c in OPTIONAL_COLS:
            if c not in df.columns:
                df[c] = ""
        if (df["source"] == "").all():
            df["source"] = "Unknown"
        df["source_file"] = os.path.basename(p)
        frames.append(df[["claim_text", "verdict", "source_file"] + OPTIONAL_COLS])
        print(f"Loaded {len(df):5d} rows from {p}")
    if not frames:
        raise ValueError("No input rows found. Add CSVs to data/raw/ first.")
    return pd.concat(frames, ignore_index=True)


def _standardise_source(s) -> str:
    s0 = str(s).strip()
    low = s0.lower()
    if "dubawa" in low:
        return "Dubawa"
    if "africa check" in low or "africacheck" in low:
        return "Africa Check"
    return s0 or "Unknown"


# ---------------------------------------------------------------------------
# 6. Split
# ---------------------------------------------------------------------------
def stratified_split(df, log):
    def _split(data, test_size, key):
        return train_test_split(data, test_size=test_size,
                                stratify=data[key] if key else None,
                                random_state=RANDOM_STATE)

    df = df.copy()
    df["_strat_ls"] = df["label"] + "|" + df["source"]
    for key, desc in [("_strat_ls", "label + source"), ("label", "label"), (None, "random")]:
        try:
            train, temp = _split(df, 0.20, key)
            val, test = _split(temp, 0.50, key)
            log["split_stratified_on"] = desc
            print(f"Split stratified on: {desc}")
            return [d.drop(columns=["_strat_ls"]) for d in (train, val, test)]
        except ValueError:
            continue
    raise ValueError("Could not split the data - too few rows.")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def run(inputs=None, raw_dir="data/raw", outdir="data", include_sample=False, stem=True):
    os.makedirs(outdir, exist_ok=True)
    log = {"run_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    paths = resolve_inputs(inputs, raw_dir, include_sample)
    log["input_files"] = [os.path.basename(p) for p in paths]
    log["is_sample_only"] = all(os.path.basename(p) == SAMPLE_FILENAME for p in paths)

    df = load_raw(paths)
    log["rows_loaded"] = int(len(df))

    df["claim_text"] = df["claim_text"].astype(str).str.strip()
    df = df[df["claim_text"].str.len() > 0].copy()
    df["source"] = df["source"].apply(_standardise_source)

    # 1. Label harmonisation
    df["label"] = df["verdict"].apply(harmonize_label)
    unmapped = df.loc[df["label"].isna(), "verdict"].str.lower().value_counts().head(25)
    log["dropped_unverifiable_or_unmapped"] = int(df["label"].isna().sum())
    log["top_dropped_verdicts"] = {str(k)[:60]: int(v) for k, v in unmapped.items()}
    df = df.dropna(subset=["label"]).copy()
    print(f"Label harmonisation: dropped {log['dropped_unverifiable_or_unmapped']} rows "
          f"(unverifiable / unmapped). Remaining: {len(df)}")
    if len(df) == 0:
        raise ValueError("No rows left after label harmonisation. Check VERDICT_MAP.")

    # 2. Language identification pass
    df["language_tag_given"] = df["language_tag"].str.strip().str.lower()
    df["language_tag"] = df["claim_text"].apply(detect_language)
    given = df[df["language_tag_given"].isin(["en", "pcm"])]
    if len(given):
        agree = float((given["language_tag_given"] == given["language_tag"]).mean())
        log["language_id_agreement_with_given_tags"] = round(agree, 4)
        log["language_id_agreement_n"] = int(len(given))
        print(f"Language-ID agreement with provided tags: {agree:.1%} (n={len(given)})")

    # 3. Category and group tags (a provided category is kept)
    tags = df["claim_text"].apply(tag_groups).apply(pd.Series)
    provided_cat = df["category"].str.strip().str.lower()
    df["category"] = np.where(provided_cat.isin(list(CATEGORY_KEYWORDS) + ["other"]),
                              provided_cat, tags["category"])
    for c in ["region", "party_ref", "religion_ref", "ethnic_ref"]:
        df[c] = tags[c]

    # 4. Duplicates and conflicting verdicts
    before = len(df)
    df = remove_duplicates(df, log)
    print(f"Deduplication: {before} -> {len(df)} "
          f"(exact {log['exact_duplicates_removed']}, near {log['near_duplicates_removed']}, "
          f"conflicting verdicts {log['conflicting_verdicts_removed']})")

    # 5. Cleaning
    df["clean_text"] = [clean_text(t, lang=l, stem=stem)
                        for t, l in zip(df["claim_text"], df["language_tag"])]
    empty = int((df["clean_text"].str.strip() == "").sum())
    df = df[df["clean_text"].str.strip() != ""].copy()
    log["empty_after_cleaning_removed"] = empty

    log["final_rows"] = int(len(df))
    log["label_counts"] = df["label"].value_counts().to_dict()
    log["language_counts"] = df["language_tag"].value_counts().to_dict()
    log["category_counts"] = df["category"].value_counts().to_dict()
    log["source_counts"] = df["source"].value_counts().to_dict()
    log["region_counts"] = df["region"].value_counts().to_dict()
    print("\nLabels:", log["label_counts"])
    print("Languages:", log["language_counts"])
    print("Categories:", log["category_counts"])
    print("Sources:", log["source_counts"])

    if df["label"].nunique() < 2:
        raise ValueError("Only one class left - need both Real and Fake claims.")

    # 6. Split
    train, val, test = stratified_split(df, log)
    for name, part in (("train", train), ("val", val), ("test", test)):
        part = part.assign(split=name)
        part.to_csv(os.path.join(outdir, f"{name}.csv"), index=False)
    full = pd.concat([train.assign(split="train"), val.assign(split="val"),
                      test.assign(split="test")])
    os.makedirs(outdir, exist_ok=True)
    full.to_csv(os.path.join(outdir, "claims_clean.csv"), index=False)
    log["split_sizes"] = {"train": len(train), "val": len(val), "test": len(test)}
    with open(os.path.join(outdir, "preprocess_log.json"), "w") as f:
        json.dump(log, f, indent=2)
    print(f"\ntrain={len(train)} val={len(val)} test={len(test)}")
    print(f"Saved {outdir}/train.csv, val.csv, test.csv, claims_clean.csv, preprocess_log.json")
    return log


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--inputs", nargs="*", help="CSV files or globs (default: data/raw/*.csv)")
    ap.add_argument("--raw-dir", default="data/raw")
    ap.add_argument("--outdir", default="data")
    ap.add_argument("--include-sample", action="store_true")
    ap.add_argument("--no-stem", action="store_true", help="disable English stemming")
    a = ap.parse_args()
    os.makedirs(a.outdir, exist_ok=True)
    run(a.inputs, a.raw_dir, a.outdir, a.include_sample, stem=not a.no_stem)

In [ ]:
%%writefile fetch_factcheck_api.py
"""
fetch_factcheck_api.py
======================
Stage 1 data ingestion (Section 3.4.2.1): pulls fact-checked claims from the
free Google Fact Check Tools API, restricted to Dubawa and Africa Check
(the study's single documented source), and appends them to
data/raw/api_claims.csv. Safe to re-run: rows are deduplicated.

Also exposes lookup_claim(), used by the app to cross-reference a submitted
claim against existing fact-checker verdicts (FR8).

Get a free key: Google Cloud Console -> enable "Fact Check Tools API" ->
Credentials -> Create API key.

Usage:
  export GOOGLE_FACTCHECK_API_KEY=your_key
  python fetch_factcheck_api.py                     # default Nigerian topic queries
  python fetch_factcheck_api.py --queries "fuel subsidy" "INEC" --max-pages 10
"""

import argparse
import csv
import os
import sys
import time

import requests
from requests.adapters import HTTPAdapter, Retry

API_URL = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
FIELDNAMES = ["claim_id", "claim_text", "verdict", "source", "language_tag", "date", "url"]
PUBLISHER_SITES = ["dubawa.org", "africacheck.org"]
DEFAULT_QUERIES = [
    "Nigeria", "Nigerian", "Lagos", "Abuja", "INEC", "Tinubu", "naira", "CBN",
    "fuel subsidy", "Boko Haram", "bandits", "Kano", "Port Harcourt", "NCDC",
    "vaccine Nigeria", "election Nigeria", "Peter Obi", "Atiku", "EFCC", "Nigerian army",
]


def make_session():
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1.0,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=["GET"])
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.headers["User-Agent"] = "FakeNewsDetectorNG/1.0 (academic project)"
    return session


def _is_allowed_publisher(review, sites):
    pub = review.get("publisher") or {}
    hay = f"{pub.get('site', '')} {pub.get('name', '')} {review.get('url', '')}".lower()
    keys = [s.lower() for s in sites] + ["dubawa", "africa check", "africacheck"]
    return any(k in hay for k in keys)


def _rows_from_response(data, sites):
    rows = []
    for claim in data.get("claims", []) or []:
        text = (claim.get("text") or "").strip()
        for review in claim.get("claimReview", []) or []:
            if sites and not _is_allowed_publisher(review, sites):
                continue
            rating = (review.get("textualRating") or "").strip()
            publisher = (review.get("publisher") or {}).get("name") or "Unknown"
            url = review.get("url") or ""
            if text and rating:
                rows.append({
                    "claim_id": url or text[:80],
                    "claim_text": text,
                    "verdict": rating,
                    "source": publisher,
                    "language_tag": "",  # assigned by the language-ID pass in preprocess.py
                    "date": (review.get("reviewDate") or claim.get("claimDate") or "")[:10],
                    "url": url,
                })
    return rows


def fetch_claims(api_key, query=None, site=None, language=None, max_pages=5,
                 page_size=50, session=None):
    session = session or make_session()
    rows, token = [], None
    for page in range(max_pages):
        params = {"key": api_key, "pageSize": page_size}
        if query:
            params["query"] = query
        if site:
            params["reviewPublisherSiteFilter"] = site
        if language:
            params["languageCode"] = language
        if token:
            params["pageToken"] = token
        resp = session.get(API_URL, params=params, timeout=20)
        if resp.status_code == 400 and not query:
            break  # API requires a query for this combination
        resp.raise_for_status()
        data = resp.json()
        rows.extend(_rows_from_response(data, [site] if site else PUBLISHER_SITES))
        token = data.get("nextPageToken")
        if not token:
            break
        time.sleep(0.4)
    return rows


def lookup_claim(text, api_key, timeout=8, max_results=5):
    """FR8: find existing Dubawa / Africa Check verdicts for a submitted claim.
    Returns a list of dicts; raises on network/API failure (caller degrades gracefully)."""
    query = " ".join(str(text).split()[:30])[:250]
    resp = requests.get(API_URL, params={"key": api_key, "query": query, "pageSize": 20},
                        timeout=timeout)
    resp.raise_for_status()
    rows = _rows_from_response(resp.json(), PUBLISHER_SITES)
    return rows[:max_results]


def load_existing(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def dedupe(rows):
    seen, out = set(), []
    for r in rows:
        key = (r.get("claim_id"), r.get("source"))
        if key in seen:
            continue
        seen.add(key)
        out.append(r)
    return out


def run(api_key, queries=None, max_pages=5, out_path="data/raw/api_claims.csv"):
    if not api_key:
        raise ValueError("No API key. Set GOOGLE_FACTCHECK_API_KEY or pass --api-key.")
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    existing = load_existing(out_path)
    session = make_session()
    new_rows = []
    for site in PUBLISHER_SITES:
        for q in (queries or DEFAULT_QUERIES):
            try:
                got = fetch_claims(api_key, query=q, site=site, max_pages=max_pages,
                                   session=session)
            except requests.HTTPError as e:
                print(f"  [{site}] '{q}': HTTP error {e}")
                continue
            print(f"  [{site}] '{q}': {len(got)} rows")
            new_rows.extend(got)
    merged = dedupe(existing + new_rows)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(merged)
    added = len(merged) - len(existing)
    print(f"Fetched {len(new_rows)} rows, {added} new. Total in {out_path}: {len(merged)}")
    return added


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--api-key", default=os.environ.get("GOOGLE_FACTCHECK_API_KEY"))
    ap.add_argument("--queries", nargs="*", default=None)
    ap.add_argument("--max-pages", type=int, default=5)
    ap.add_argument("--out", default="data/raw/api_claims.csv")
    a = ap.parse_args()
    if not a.api_key:
        sys.exit("Error: provide --api-key or set GOOGLE_FACTCHECK_API_KEY.")
    run(a.api_key, a.queries, a.max_pages, a.out)

In [ ]:
%%writefile scrape_factcheckers.py
"""
scrape_factcheckers.py
======================
Stage 1 data ingestion (Sections 3.4.2.1 and 3.4.6): collects published
verdicts directly from the Dubawa and Africa Check archives using Requests +
BeautifulSoup, to supplement the Google Fact Check Tools API.

  Dubawa       -> WordPress REST API (dubawa.org/wp-json/wp/v2/posts). Each
                  fact-check has "Claim:" and "Verdict:" labels in the body.
  Africa Check -> Nigeria listing (africacheck.org/fact-checks?field_country_value=NG)
                  then each article page. Verdict is read from ClaimReview
                  structured data, a rating element, or the rating image.

Rows whose verdict or claim cannot be read reliably are NOT guessed. They go
to data/raw/needs_review_<site>.csv so a team member can open the article and
fill the 'verdict' column by hand from the published article (no claim is
labelled without a published verdict). preprocess.py automatically uses the
rows whose verdict has been filled in and ignores the rest.

Be polite: keep the delay (default 1.5 s) and respect each site's terms of use;
attribution is preserved in the 'source' and 'url' columns.

Usage:
  python scrape_factcheckers.py --site dubawa --max-pages 20
  python scrape_factcheckers.py --site africacheck --max-pages 30
  python scrape_factcheckers.py --site both
"""

import argparse
import csv
import json
import os
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

UA = {"User-Agent": "FakeNewsDetectorNG/1.0 (academic project; contact: project team)"}
FIELDNAMES = ["claim_id", "claim_text", "verdict", "source", "language_tag", "date", "url"]
REVIEW_FIELDS = ["url", "title", "claim_guess", "verdict_guess", "verdict", "source", "date"]

VERDICT_WORDS = [
    "mostly false", "mostly true", "mostly correct", "partly false", "partly true",
    "half true", "false", "true", "misleading", "incorrect", "correct", "unproven",
    "unverifiable", "unverified", "exaggerated", "understated", "fake", "scam",
    "fabricated", "manipulated", "satire",
]
_VERDICT_RE = re.compile(r"\b(" + "|".join(re.escape(w) for w in VERDICT_WORDS) + r")\b", re.I)
NIGERIA_RE = re.compile(
    r"\b(nigeria|nigerian|nigerians|naija|lagos|abuja|kano|port harcourt|ibadan|kaduna|"
    r"enugu|inec|tinubu|buhari|naira|cbn|efcc|ncdc|boko haram)\b", re.I)


def _get(session, url, **kw):
    r = session.get(url, headers=UA, timeout=25, **kw)
    r.raise_for_status()
    return r


def _first_verdict(text):
    m = _VERDICT_RE.search(text or "")
    return m.group(1).title() if m else ""


# ---------------------------------------------------------------------------
# Dubawa
# ---------------------------------------------------------------------------
def parse_dubawa_post(title_html, content_html):
    """Return (claim, verdict) from a Dubawa post. Empty strings if not found."""
    soup = BeautifulSoup(content_html or "", "html.parser")
    text = soup.get_text("\n")
    text = re.sub(r"[ \t\xa0]+", " ", text)
    claim, verdict = "", ""
    m = re.search(r"Claim\s*:\s*(.+?)(?=\n\s*\n|Verdict\s*:|Full\s*Text|$)", text, re.I | re.S)
    if m:
        claim = " ".join(m.group(1).split())
    m = re.search(r"Verdict\s*:\s*([^\n]{1,200})", text, re.I)
    if m:
        verdict = _first_verdict(m.group(1))
    title = BeautifulSoup(title_html or "", "html.parser").get_text(" ").strip()
    if not verdict:
        m = re.match(r"^\s*(" + "|".join(re.escape(w) for w in VERDICT_WORDS) + r")\s*[!:.\-]",
                     title, re.I)
        if m:
            verdict = m.group(1).title()
    return claim, verdict, title, text


def scrape_dubawa(max_pages=10, per_page=50, delay=1.5, nigeria_only=True, search=None):
    session = requests.Session()
    rows, review = [], []
    for page in range(1, max_pages + 1):
        params = {"per_page": per_page, "page": page,
                  "_fields": "id,link,title,date,content"}
        if search:
            params["search"] = search
        try:
            r = _get(session, "https://dubawa.org/wp-json/wp/v2/posts", params=params)
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 400:
                break  # past the last page
            raise
        posts = r.json()
        if not posts:
            break
        for p in posts:
            claim, verdict, title, body = parse_dubawa_post(
                (p.get("title") or {}).get("rendered", ""),
                (p.get("content") or {}).get("rendered", ""))
            if nigeria_only and not NIGERIA_RE.search(f"{title} {body}"):
                continue
            if "claim" not in body.lower() and not verdict:
                continue  # not a fact-check (news, training, reports)
            link, date = p.get("link", ""), (p.get("date") or "")[:10]
            if claim and verdict:
                rows.append({"claim_id": link, "claim_text": claim, "verdict": verdict,
                             "source": "Dubawa", "language_tag": "", "date": date, "url": link})
            else:
                review.append({"url": link, "title": title, "claim_guess": claim or title,
                               "verdict_guess": verdict, "verdict": "", "source": "Dubawa",
                               "date": date})
        total_pages = int(r.headers.get("X-WP-TotalPages", page))
        print(f"  Dubawa page {page}/{total_pages}: {len(rows)} usable, {len(review)} to review")
        if page >= total_pages:
            break
        time.sleep(delay)
    return rows, review


# ---------------------------------------------------------------------------
# Africa Check
# ---------------------------------------------------------------------------
AC_LIST = ("https://africacheck.org/fact-checks?field_article_type_value=All"
           "&field_rated_value=All&field_country_value=NG&sort_bef_combine=created_DESC&page={page}")
AC_LINK_RE = re.compile(r"^/(?:index\.php/|index\.ph%70/)?fact-checks/[a-z0-9\-]+/[a-z0-9\-]+/?$")


def parse_africacheck_listing(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].split("?")[0].split("#")[0]
        if href.startswith("https://africacheck.org"):
            href = href[len("https://africacheck.org"):]
        if AC_LINK_RE.match(href):
            href = href.replace("/index.ph%70", "").replace("/index.php", "")
            full = urljoin("https://africacheck.org", href)
            if full not in links:
                links.append(full)
    return links


def _claimreview_from_jsonld(soup):
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except (json.JSONDecodeError, TypeError):
            continue
        stack = data if isinstance(data, list) else [data]
        while stack:
            node = stack.pop()
            if isinstance(node, dict):
                if node.get("@type") == "ClaimReview":
                    rating = node.get("reviewRating") or {}
                    return (node.get("claimReviewed") or "",
                            rating.get("alternateName") or rating.get("name") or "")
                stack.extend(v for v in node.values() if isinstance(v, (dict, list)))
            elif isinstance(node, list):
                stack.extend(node)
    return "", ""


def parse_africacheck_article(html_text):
    """Return (claim, verdict, title, date)."""
    soup = BeautifulSoup(html_text, "html.parser")
    title = (soup.find("h1").get_text(" ", strip=True) if soup.find("h1") else "")
    date = ""
    t = soup.find("time")
    if t is not None:
        date = (t.get("datetime") or t.get_text(strip=True))[:10]

    claim, verdict = _claimreview_from_jsonld(soup)
    verdict = _first_verdict(verdict) if verdict else ""

    if not verdict:  # rating element (class names containing rating/verdict)
        for el in soup.find_all(class_=re.compile(r"(rating|verdict)", re.I)):
            verdict = _first_verdict(el.get_text(" ", strip=True))
            if verdict:
                break
    if not verdict:  # rating image, e.g. ..._False.png / alt="Incorrect"
        for img in soup.find_all("img"):
            hay = f"{img.get('alt', '')} {os.path.basename(img.get('src', ''))}"
            hay = re.sub(r"[_\-]", " ", hay)
            if re.search(r"rating|verdict|rated|false|incorrect|correct|misleading", hay, re.I):
                verdict = _first_verdict(hay)
                if verdict:
                    break

    if not claim:  # "IN SHORT:" summary or meta description
        for el in soup.find_all(["p", "div", "strong"]):
            txt = el.get_text(" ", strip=True)
            if txt.upper().startswith("IN SHORT"):
                claim = re.sub(r"^IN SHORT\s*:?\s*", "", txt, flags=re.I)
                break
    if not claim:
        meta = soup.find("meta", attrs={"name": "description"}) or \
            soup.find("meta", attrs={"property": "og:description"})
        claim = meta.get("content", "") if meta else ""
    return " ".join(claim.split()), verdict, title, date


def scrape_africacheck(max_pages=10, delay=1.5):
    session = requests.Session()
    rows, review, seen = [], [], set()
    for page in range(max_pages):
        r = _get(session, AC_LIST.format(page=page))
        links = [l for l in parse_africacheck_listing(r.text) if l not in seen]
        if not links:
            break
        for link in links:
            seen.add(link)
            time.sleep(delay)
            try:
                art = _get(session, link)
            except requests.RequestException as e:
                print(f"    skip {link}: {e}")
                continue
            claim, verdict, title, date = parse_africacheck_article(art.text)
            is_report = "/reports/" in link  # reports check many claims at once
            if claim and verdict and not is_report:
                rows.append({"claim_id": link, "claim_text": claim, "verdict": verdict,
                             "source": "Africa Check", "language_tag": "", "date": date,
                             "url": link})
            else:
                review.append({"url": link, "title": title, "claim_guess": claim or title,
                               "verdict_guess": verdict, "verdict": "",
                               "source": "Africa Check", "date": date})
        print(f"  Africa Check page {page}: {len(rows)} usable, {len(review)} to review")
        time.sleep(delay)
    return rows, review


# ---------------------------------------------------------------------------
def _write(path, rows, fields):
    existing = []
    if os.path.exists(path) and os.path.getsize(path) > 0:
        with open(path, newline="", encoding="utf-8") as f:
            existing = list(csv.DictReader(f))
    seen, merged = set(), []
    for r in existing + rows:
        if r.get("url") in seen:
            continue
        seen.add(r.get("url"))
        merged.append(r)
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(merged)
    return len(merged)


def run(site="both", max_pages=10, delay=1.5, raw_dir="data/raw"):
    if site in ("dubawa", "both"):
        rows, review = scrape_dubawa(max_pages=max_pages, delay=delay)
        n = _write(os.path.join(raw_dir, "dubawa_claims.csv"), rows, FIELDNAMES)
        m = _write(os.path.join(raw_dir, "needs_review_dubawa.csv"), review, REVIEW_FIELDS)
        print(f"Dubawa: {n} labelled claims saved, {m} need manual review")
    if site in ("africacheck", "both"):
        rows, review = scrape_africacheck(max_pages=max_pages, delay=delay)
        n = _write(os.path.join(raw_dir, "africacheck_claims.csv"), rows, FIELDNAMES)
        m = _write(os.path.join(raw_dir, "needs_review_africacheck.csv"), review, REVIEW_FIELDS)
        print(f"Africa Check: {n} labelled claims saved, {m} need manual review")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--site", choices=["dubawa", "africacheck", "both"], default="both")
    ap.add_argument("--max-pages", type=int, default=10)
    ap.add_argument("--delay", type=float, default=1.5)
    ap.add_argument("--raw-dir", default="data/raw")
    a = ap.parse_args()
    run(a.site, a.max_pages, a.delay, a.raw_dir)

In [ ]:
%%writefile train_model.py
"""
train_model.py
==============
Stages 3-4 of the pipeline (Sections 3.4.4 - 3.4.5).

  * TF-IDF (unigrams + bigrams) fitted on the TRAINING split only
  * Three classifiers: Multinomial Naive Bayes, Logistic Regression, Random Forest
  * Hyperparameters tuned with stratified k-fold cross-validation on the
    training split; SMOTE is applied INSIDE each fold and to the training
    split only, never to validation or test (Section 3.4.2.4)
  * Metrics per split: accuracy, precision, recall, F1 (Fake class),
    macro-F1, AUC-ROC, Brier score, confusion matrix, false-positive rate
    (genuine claims wrongly flagged as Fake). Train accuracy is reported so
    the train-test gap can flag overfitting (3.4.5)
  * Disaggregated (bias-audit) results by language, category, source,
    region, party reference, religion reference and ethnic reference
  * Figures for Chapter Four saved to models/figures/

Outputs (models/):
  vectorizer.pkl, naive_bayes.pkl, logistic_regression.pkl, random_forest.pkl,
  best_model_name.txt, metrics_report.json, metrics_summary.csv,
  disaggregated_report.csv, training_meta.json, figures/*.png

Usage:
  python train_model.py --datadir data --outdir models
"""

import argparse
import json
import os
import platform
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import MultinomialNB

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
LABELS = ["Real", "Fake"]
POS = "Fake"
GROUP_COLUMNS = ["language_tag", "category", "source", "region", "party_ref",
                 "religion_ref", "ethnic_ref"]
DISPLAY = {"naive_bayes": "Naive Bayes", "logistic_regression": "Logistic Regression",
           "random_forest": "Random Forest"}


def load_split(datadir, name):
    path = os.path.join(datadir, f"{name}.csv")
    df = pd.read_csv(path, keep_default_na=False)
    missing = {"clean_text", "label"} - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing {missing}. Run preprocess.py first.")
    for c in GROUP_COLUMNS:
        if c not in df.columns:
            df[c] = "Unknown"
    df["clean_text"] = df["clean_text"].fillna("").astype(str)
    return df.reset_index(drop=True)


def p_fake(model, X):
    classes = list(model.classes_)
    return model.predict_proba(X)[:, classes.index(POS)]


def compute_metrics(y_true, y_pred, pf):
    y_true = np.asarray(y_true)
    y_bin = (y_true == POS).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    tn, fp = cm[0, 0], cm[0, 1]  # row 0 = actual Real
    out = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_fake": float(precision_score(y_true, y_pred, pos_label=POS, zero_division=0)),
        "recall_fake": float(recall_score(y_true, y_pred, pos_label=POS, zero_division=0)),
        "f1_fake": float(f1_score(y_true, y_pred, pos_label=POS, zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", labels=LABELS,
                                   zero_division=0)),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else None,
        "brier_score": float(brier_score_loss(y_bin, pf)),
        "roc_auc": None,
        "confusion_matrix": cm.tolist(),
    }
    if len(set(y_bin)) == 2:
        out["roc_auc"] = float(roc_auc_score(y_bin, pf))
    return out


def build_candidates(minority_train):
    # SMOTE needs k_neighbors < minority samples inside each CV training fold
    k = max(1, min(5, int(minority_train * 0.6) - 1))
    use_smote = minority_train >= 3

    def pipe(clf):
        steps = [("smote", SMOTE(k_neighbors=k, random_state=RANDOM_STATE))] if use_smote else []
        return ImbPipeline(steps + [("clf", clf)])

    return {
        "naive_bayes": (pipe(MultinomialNB()), {"clf__alpha": [0.1, 0.5, 1.0]}),
        "logistic_regression": (
            pipe(LogisticRegression(solver="liblinear", max_iter=2000,
                                    random_state=RANDOM_STATE)),
            {"clf__C": [0.1, 1, 10]}),
        "random_forest": (
            pipe(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
            {"clf__n_estimators": [100, 200, 300], "clf__max_depth": [None, 30]}),
    }, use_smote, k


def disaggregate(model_key, split, df, y_pred, pf):
    rows = []
    for col in GROUP_COLUMNS:
        for grp, idx in df.groupby(col).groups.items():
            idx = np.asarray(list(idx))
            m = compute_metrics(df.loc[idx, "label"].values, y_pred[idx], pf[idx])
            rows.append({
                "model": DISPLAY[model_key], "split": split, "group_column": col,
                "group": grp, "n": m["n"], "accuracy": m["accuracy"],
                "f1_macro": m["f1_macro"], "precision_fake": m["precision_fake"],
                "recall_fake": m["recall_fake"], "false_positive_rate": m["false_positive_rate"],
                "small_sample_warning": m["n"] < 10,
            })
    return rows


def make_figures(outdir, results, test_df, test_scores):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns

    figdir = os.path.join(outdir, "figures")
    os.makedirs(figdir, exist_ok=True)
    sns.set_theme(style="whitegrid", context="notebook")
    colors = ["#2a6f97", "#c9732b", "#5a8f3d"]
    keys = list(results)

    # 1. Confusion matrices (test set)
    fig, axes = plt.subplots(1, len(keys), figsize=(4.4 * len(keys), 4))
    for ax, k in zip(np.atleast_1d(axes), keys):
        cm = np.array(results[k]["test"]["confusion_matrix"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=LABELS, yticklabels=LABELS)
        ax.set_title(DISPLAY[k])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    fig.suptitle("Confusion matrices - test set")
    fig.tight_layout()
    fig.savefig(os.path.join(figdir, "confusion_matrices_test.png"), dpi=150)
    plt.close(fig)

    # 2. ROC curves (test set)
    y_bin = (test_df["label"].values == POS).astype(int)
    if len(set(y_bin)) == 2:
        fig, ax = plt.subplots(figsize=(5.5, 5))
        for k, c in zip(keys, colors):
            fpr, tpr, _ = roc_curve(y_bin, test_scores[k])
            auc = results[k]["test"]["roc_auc"]
            ax.plot(fpr, tpr, color=c, lw=2, label=f"{DISPLAY[k]} (AUC {auc:.3f})")
        ax.plot([0, 1], [0, 1], ls="--", color="#999999", lw=1)
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate (Fake recall)")
        ax.set_title("ROC curves - test set")
        ax.legend(loc="lower right", frameon=False)
        fig.tight_layout()
        fig.savefig(os.path.join(figdir, "roc_curves_test.png"), dpi=150)
        plt.close(fig)

    # 3. Metric comparison (test set)
    metrics = ["accuracy", "precision_fake", "recall_fake", "f1_fake", "f1_macro", "roc_auc"]
    names = ["Accuracy", "Precision (Fake)", "Recall (Fake)", "F1 (Fake)", "Macro F1", "AUC-ROC"]
    x = np.arange(len(metrics))
    w = 0.8 / len(keys)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for i, (k, c) in enumerate(zip(keys, colors)):
        vals = [results[k]["test"][m] or 0 for m in metrics]
        ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=DISPLAY[k], color=c)
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, 1.05)
    ax.set_title("Classifier comparison - test set")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    fig.tight_layout()
    fig.savefig(os.path.join(figdir, "metric_comparison_test.png"), dpi=150)
    plt.close(fig)

    # 4. Train vs test accuracy (overfitting check)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(len(keys))
    ax.bar(x - 0.18, [results[k]["train"]["accuracy"] for k in keys], 0.36,
           label="Train", color="#9db4c0")
    ax.bar(x + 0.18, [results[k]["test"]["accuracy"] for k in keys], 0.36,
           label="Test", color="#2a6f97")
    ax.set_xticks(x)
    ax.set_xticklabels([DISPLAY[k] for k in keys])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Accuracy")
    ax.set_title("Train vs test accuracy (overfitting check)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(figdir, "train_vs_test_accuracy.png"), dpi=150)
    plt.close(fig)

    # 5. Macro-F1 by language (test set)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    langs = sorted(test_df["language_tag"].unique())
    x = np.arange(len(langs))
    for i, (k, c) in enumerate(zip(keys, colors)):
        vals = [results[k]["test_by_language"].get(l, {}).get("f1_macro", 0) for l in langs]
        ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=DISPLAY[k], color=c)
    ax.set_xticks(x)
    ax.set_xticklabels({"en": "English", "pcm": "Pidgin / code-switched"}.get(l, l)
                       for l in langs)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Macro F1")
    ax.set_title("Performance by language - test set")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(figdir, "f1_by_language_test.png"), dpi=150)
    plt.close(fig)
    return figdir


def run(datadir="data", outdir="models"):
    os.makedirs(outdir, exist_ok=True)
    train, val, test = (load_split(datadir, s) for s in ("train", "val", "test"))
    print(f"train={len(train)} val={len(val)} test={len(test)}")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000,
                                 min_df=2 if len(train) >= 200 else 1, sublinear_tf=True)
    X_train = vectorizer.fit_transform(train["clean_text"])   # fit on train ONLY
    X_val = vectorizer.transform(val["clean_text"])
    X_test = vectorizer.transform(test["clean_text"])
    joblib.dump(vectorizer, os.path.join(outdir, "vectorizer.pkl"))
    print(f"TF-IDF vocabulary: {len(vectorizer.vocabulary_)} features")

    counts = train["label"].value_counts()
    minority = int(counts.min())
    n_splits = max(2, min(5, minority))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    candidates, use_smote, k = build_candidates(minority)

    smote_info = {"applied": use_smote, "k_neighbors": k,
                  "train_counts_before": {k_: int(v) for k_, v in counts.items()}}
    if use_smote:
        _, y_res = SMOTE(k_neighbors=k, random_state=RANDOM_STATE).fit_resample(
            X_train, train["label"])
        smote_info["train_counts_after"] = {k_: int(v) for k_, v in
                                            pd.Series(y_res).value_counts().items()}
        print(f"SMOTE on training split only: {smote_info['train_counts_before']} -> "
              f"{smote_info['train_counts_after']}")
    else:
        print("SMOTE skipped: minority class too small (<3).")

    results, disagg, test_scores = {}, [], {}
    for key, (pipe, grid) in candidates.items():
        print(f"\n=== {DISPLAY[key]}: {n_splits}-fold CV grid search ===")
        search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=-1)
        search.fit(X_train, train["label"])
        model = search.best_estimator_.named_steps["clf"]  # trained on SMOTE-balanced train
        print(f"Best params: {search.best_params_}  CV macro-F1={search.best_score_:.4f}")

        res = {"best_params": {k_.replace("clf__", ""): v for k_, v in search.best_params_.items()},
               "cv_f1_macro": float(search.best_score_)}
        for split, df, X in (("train", train, X_train), ("val", val, X_val), ("test", test, X_test)):
            pred = model.predict(X)
            pf = p_fake(model, X)
            res[split] = compute_metrics(df["label"].values, pred, pf)
            if split in ("val", "test"):
                disagg.extend(disaggregate(key, split, df, pred, pf))
            if split == "test":
                test_scores[key] = pf
                res["test_by_language"] = {
                    lang: compute_metrics(df.loc[idx, "label"].values, pred[np.asarray(list(idx))],
                                          pf[np.asarray(list(idx))])
                    for lang, idx in df.groupby("language_tag").groups.items()}
        t = res["test"]
        auc = f"{t['roc_auc']:.3f}" if t["roc_auc"] is not None else "n/a"
        print(f"  train acc={res['train']['accuracy']:.3f} | test acc={t['accuracy']:.3f} "
              f"P={t['precision_fake']:.3f} R={t['recall_fake']:.3f} F1={t['f1_fake']:.3f} "
              f"macroF1={t['f1_macro']:.3f} AUC={auc} Brier={t['brier_score']:.3f}")
        for lang, m in res["test_by_language"].items():
            print(f"    [{lang}] n={m['n']} macroF1={m['f1_macro']:.3f} acc={m['accuracy']:.3f}")
        results[key] = res
        joblib.dump(model, os.path.join(outdir, f"{key}.pkl"), compress=3)  # keeps files under GitHub upload limit

    best = max(results, key=lambda k_: results[k_]["val"]["f1_macro"])
    print(f"\nBest model by validation macro-F1: {DISPLAY[best]} "
          f"(test macro-F1 {results[best]['test']['f1_macro']:.4f}, unbiased estimate)")
    with open(os.path.join(outdir, "best_model_name.txt"), "w") as f:
        f.write(best)

    report = {"best_model_by_val_f1_macro": best, "models": results}
    with open(os.path.join(outdir, "metrics_report.json"), "w") as f:
        json.dump(report, f, indent=2, default=str)

    summary = []
    for k_, r in results.items():
        for split in ("train", "val", "test"):
            m = r[split]
            summary.append({"model": DISPLAY[k_], "split": split, "n": m["n"],
                            "accuracy": m["accuracy"], "precision_fake": m["precision_fake"],
                            "recall_fake": m["recall_fake"], "f1_fake": m["f1_fake"],
                            "f1_macro": m["f1_macro"], "roc_auc": m["roc_auc"],
                            "brier_score": m["brier_score"],
                            "false_positive_rate": m["false_positive_rate"]})
    pd.DataFrame(summary).round(4).to_csv(os.path.join(outdir, "metrics_summary.csv"), index=False)
    pd.DataFrame(disagg).round(4).to_csv(os.path.join(outdir, "disaggregated_report.csv"),
                                         index=False)

    prep_log = {}
    p = os.path.join(datadir, "preprocess_log.json")
    if os.path.exists(p):
        with open(p) as f:
            prep_log = json.load(f)
    meta = {
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "n_train": len(train), "n_val": len(val), "n_test": len(test),
        "cv_folds": n_splits, "smote": smote_info,
        "is_sample_only": bool(prep_log.get("is_sample_only", False)),
        "input_files": prep_log.get("input_files", []),
        "python": platform.python_version(), "scikit_learn": sklearn.__version__,
        "decision_threshold_p_real": 0.5,
    }
    with open(os.path.join(outdir, "training_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    figdir = make_figures(outdir, results, test, test_scores)
    print(f"\nSaved models, reports and figures to {outdir}/ (figures in {figdir}/)")
    return report


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--datadir", default="data")
    ap.add_argument("--outdir", default="models")
    a = ap.parse_args()
    run(a.datadir, a.outdir)

In [ ]:
%%writefile app.py
"""
app.py - Fake News Detector for Nigerian Online Media (Streamlit)
=================================================================
Stage 5 / Presentation tier (Sections 3.3.2, 3.3.5 - 3.3.7).

  Check a claim   FR1 text, FR2 URL, FR3 preprocessing, FR4 label + 0-1 score,
                  FR7 pasted WhatsApp text, FR8 cross-reference, FR9 logging,
                  OFR1 LIME explanation, NFR6 graceful degradation
  Search claims   FR6 keyword / topic / source search
  Admin           FR10 retrain with new data, FR11 disaggregated reports,
                  NFR8 log-based auditing, NFR10 data refresh

Run locally:   streamlit run app.py
Secrets (optional), in .streamlit/secrets.toml or Streamlit Cloud settings:
  GOOGLE_FACTCHECK_API_KEY = "..."   enables live fact-checker cross-reference
  ADMIN_PASSWORD = "..."             enables the Admin tab
"""

import contextlib
import hmac
import io
import json
import os
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
import streamlit as st

from preprocess import clean_text, detect_language, scrub_pii, tag_groups

BASE = Path(__file__).resolve().parent
MODEL_DIR = BASE / "models"
DATA_DIR = BASE / "data"
RAW_DIR = DATA_DIR / "raw"
LOG_DIR = BASE / "logs"
LOG_DB = LOG_DIR / "classifications.db"
THRESHOLD = 0.5  # P(Real) >= 0.5 -> Real (fixed decision threshold, FR4)

MODEL_FILES = {
    "naive_bayes": "Naive Bayes",
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
}
LANG_NAMES = {"en": "English", "pcm": "Nigerian Pidgin / code-switched"}

st.set_page_config(page_title="Fake News Detector - Nigeria", layout="centered")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def get_secret(name):
    try:
        if name in st.secrets:
            return str(st.secrets[name])
    except Exception:
        pass
    return os.environ.get(name, "")


def _mtime(path):
    return path.stat().st_mtime if path.exists() else 0


@st.cache_resource(show_spinner="Loading models...")
def load_artifacts(_stamp):
    vec_path = MODEL_DIR / "vectorizer.pkl"
    if not vec_path.exists():
        return None
    vectorizer = joblib.load(vec_path)
    models = {}
    for key in MODEL_FILES:
        p = MODEL_DIR / f"{key}.pkl"
        if p.exists():
            models[key] = joblib.load(p)
    best = (MODEL_DIR / "best_model_name.txt")
    best = best.read_text().strip() if best.exists() else next(iter(models), None)
    meta_path = MODEL_DIR / "training_meta.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    return {"vectorizer": vectorizer, "models": models, "best": best, "meta": meta}


@st.cache_data(show_spinner=False)
def load_dataset(_stamp):
    p = DATA_DIR / "claims_clean.csv"
    if not p.exists():
        return pd.DataFrame()
    return pd.read_csv(p, keep_default_na=False)


@st.cache_resource(show_spinner=False)
def dataset_matrix(_stamp, _vectorizer):
    ds = load_dataset(_mtime(DATA_DIR / "claims_clean.csv"))
    if ds.empty:
        return None
    return _vectorizer.transform(ds["clean_text"].astype(str))


def p_real(model, X):
    return model.predict_proba(X)[:, list(model.classes_).index("Real")]


# ---- FR9 logging ----
def _db():
    LOG_DIR.mkdir(exist_ok=True)
    con = sqlite3.connect(LOG_DB)
    con.execute("""CREATE TABLE IF NOT EXISTS classifications (
        id INTEGER PRIMARY KEY AUTOINCREMENT, created_at TEXT, input_type TEXT,
        submitted_text TEXT, url TEXT, language_tag TEXT, category TEXT, region TEXT,
        party_ref TEXT, religion_ref TEXT, ethnic_ref TEXT, model TEXT,
        label TEXT, score_p_real REAL, explanation TEXT)""")
    return con


def log_event(row):
    try:
        with contextlib.closing(_db()) as con:
            con.execute("""INSERT INTO classifications (created_at, input_type, submitted_text,
                url, language_tag, category, region, party_ref, religion_ref, ethnic_ref,
                model, label, score_p_real, explanation)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
                        (row["created_at"], row["input_type"], row["submitted_text"],
                         row["url"], row["language_tag"], row["category"], row["region"],
                         row["party_ref"], row["religion_ref"], row["ethnic_ref"],
                         row["model"], row["label"], row["score_p_real"], row["explanation"]))
            con.commit()
    except sqlite3.Error as e:
        st.caption(f"Could not write to the classification log: {e}")


def read_log():
    if not LOG_DB.exists():
        return pd.DataFrame()
    with contextlib.closing(_db()) as con:
        return pd.read_sql_query("SELECT * FROM classifications ORDER BY id DESC", con)


# ---- FR2 URL input ----
def fetch_article(url):
    from bs4 import BeautifulSoup
    r = requests.get(url, timeout=12, headers={"User-Agent": "Mozilla/5.0 (FakeNewsDetectorNG)"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for t in soup(["script", "style", "nav", "footer", "header", "aside"]):
        t.decompose()

    def meta(prop):
        m = soup.find("meta", attrs={"property": prop}) or soup.find("meta", attrs={"name": prop})
        return (m.get("content") or "").strip() if m else ""

    title = meta("og:title") or (soup.h1.get_text(" ", strip=True) if soup.h1 else "") or \
        (soup.title.get_text(strip=True) if soup.title else "")
    desc = meta("og:description") or meta("description")
    paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paras = [p for p in paras if len(p) > 60][:3]
    return title, desc, paras


# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
art = load_artifacts(_mtime(MODEL_DIR / "vectorizer.pkl"))
st.title("Fake News Detector")
st.caption("Nigerian online media - English and Nigerian Pidgin. "
           "TF-IDF with Naive Bayes, Logistic Regression and Random Forest.")

if art is None or not art["models"]:
    st.error("No trained model found in models/. Run `python preprocess.py` then "
             "`python train_model.py`, and commit the models/ folder.")
    st.stop()

meta = art["meta"]
if meta.get("is_sample_only"):
    st.warning("Demo mode: these models were trained on the small sample file only, so "
               "predictions are not meaningful yet. Retrain on the full Dubawa / Africa Check "
               "dataset before evaluation.")

def run_check(mode, claim_in, url_in, model_key, show_lime, cross_ref):
    text, input_type = claim_in.strip(), "text"
    if mode == "Article URL":
        input_type = "url"
        if not url_in.strip().lower().startswith(("http://", "https://")):
            st.warning("Enter a full URL starting with http:// or https://")
            return
        try:
            with st.spinner("Fetching article..."):
                title, desc, paras = fetch_article(url_in.strip())
        except requests.RequestException as e:
            st.error(f"Could not fetch that page ({e.__class__.__name__}). "
                     "Paste the headline as text instead.")
            return
        text = " ".join(x for x in [title, desc] if x) or " ".join(paras)
        with st.expander("Text extracted from the article"):
            st.write(f"**Headline:** {title or '-'}")
            st.write(f"**Summary:** {desc or '-'}")
            for p in paras:
                st.write(p)

    if not text:
        st.warning("Please enter a claim.")
        return

    lang = detect_language(text)
    cleaned = clean_text(text, lang=lang)
    if not cleaned:
        st.warning("No usable content words after preprocessing. Try a longer claim.")
        return

    vec = art["vectorizer"].transform([cleaned])
    model = art["models"][model_key]
    score = float(p_real(model, vec)[0])
    label = "Real" if score >= THRESHOLD else "Fake"
    groups = tag_groups(text)

    st.divider()
    r1, r2 = st.columns(2)
    r1.metric("Prediction", "Likely genuine" if label == "Real" else "Likely fake")
    r2.metric("Credibility score", f"{score:.2f}")
    st.progress(score)
    st.caption("0 = highly likely fake, 1 = highly likely genuine. Threshold 0.5. "
               "This is a model score, not a verdict - see the Brier score in Admin "
               "for how well calibrated it is.")
    st.caption(f"Language: {LANG_NAMES.get(lang, lang)}  |  Topic: {groups['category']}  |  "
               f"Region: {groups['region']}  |  Model: {MODEL_FILES[model_key]}")

    with st.expander("Compare all three classifiers"):
        rows = []
        for k, m in art["models"].items():
            s = float(p_real(m, vec)[0])
            rows.append({"Classifier": MODEL_FILES[k],
                         "Prediction": "Real" if s >= THRESHOLD else "Fake",
                         "Credibility score": round(s, 3)})
        st.dataframe(pd.DataFrame(rows), hide_index=True, use_container_width=True)

    # FR8 - cross-reference with existing fact-checker verdicts
    if cross_ref:
        st.subheader("Existing fact-checks")
        ds = load_dataset(_mtime(DATA_DIR / "claims_clean.csv"))
        X_ds = dataset_matrix(_mtime(DATA_DIR / "claims_clean.csv"), art["vectorizer"])
        found = False
        if X_ds is not None and X_ds.shape[0]:
            sims = (X_ds @ vec.T).toarray().ravel()
            top = np.argsort(sims)[::-1][:3]
            hits = [(i, sims[i]) for i in top if sims[i] >= 0.35]
            if hits:
                found = True
                st.write("Similar claims already checked by Dubawa / Africa Check:")
                for i, s in hits:
                    row = ds.iloc[i]
                    link = f" - [source]({row['url']})" if str(row.get("url", "")).startswith("http") else ""
                    st.write(f"- **{row['verdict']}** ({row['source']}, similarity {s:.2f}): "
                             f"{row['claim_text']}{link}")
        api_key = get_secret("GOOGLE_FACTCHECK_API_KEY")
        if api_key:
            try:
                from fetch_factcheck_api import lookup_claim
                live = lookup_claim(text, api_key)
                if live:
                    found = True
                    st.write("Live results from the Google Fact Check Tools API:")
                    for r_ in live:
                        st.write(f"- **{r_['verdict']}** ({r_['source']}): "
                                 f"{r_['claim_text']} - [read]({r_['url']})")
            except Exception:
                st.info("Live fact-check lookup is unavailable right now. "
                        "The model result above is unaffected.")
        if not found:
            st.caption("No matching fact-check found. The model prediction stands on its own.")

    # OFR1 - LIME explanation
    explanation = []
    if show_lime:
        try:
            from lime.lime_text import LimeTextExplainer
            with st.spinner("Generating explanation..."):
                explainer = LimeTextExplainer(class_names=list(model.classes_),
                                              random_state=42)

                def predict_fn(texts):
                    return model.predict_proba(art["vectorizer"].transform(
                        [clean_text(t, lang=lang) for t in texts]))

                target = list(model.classes_).index(label)
                n_feat = min(10, max(3, len(text.split())))
                exp = explainer.explain_instance(text, predict_fn, num_features=n_feat,
                                                 labels=(target,), num_samples=1000)
                explanation = exp.as_list(label=target)
            st.subheader("Why the model decided this")
            if explanation:
                df_exp = pd.DataFrame(explanation, columns=["Word", "Weight"])
                df_exp["Effect"] = np.where(df_exp["Weight"] > 0,
                                            f"pushes towards {label}",
                                            f"pushes away from {label}")
                st.dataframe(df_exp.round(3), hide_index=True, use_container_width=True)
                st.bar_chart(df_exp.set_index("Word")["Weight"], horizontal=True)
            else:
                st.caption("No influential words found.")
        except ImportError:
            st.info("LIME is not installed, so no explanation is shown.")

    log_event({
        "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "input_type": input_type, "submitted_text": scrub_pii(text)[:2000],
        "url": url_in.strip(), "language_tag": lang, **groups,
        "model": MODEL_FILES[model_key], "label": label, "score_p_real": score,
        "explanation": json.dumps(explanation),
    })
    st.caption("Decision-support tool only. It does not remove or censor content; "
               "check important claims with Dubawa or Africa Check.")


tab_check, tab_search, tab_about, tab_admin = st.tabs(
    ["Check a claim", "Search claims", "About", "Admin"])

# ---------------------------------------------------------------------------
# Check a claim
# ---------------------------------------------------------------------------
with tab_check:
    model_keys = list(art["models"])
    default_idx = model_keys.index(art["best"]) if art["best"] in model_keys else 0

    mode = st.radio("Input", ["Paste text", "Article URL"], horizontal=True,
                    help="Text can come from anywhere, including WhatsApp forwards.")
    with st.form("check_form"):
        if mode == "Paste text":
            claim_in = st.text_area("Claim, headline or message", height=130,
                                    placeholder="e.g. Dem say CBN don recall all old naira notes")
            url_in = ""
        else:
            url_in = st.text_input("Article URL", placeholder="https://...")
            claim_in = ""
        c1, c2 = st.columns(2)
        model_key = c1.selectbox("Classifier", model_keys, index=default_idx,
                                 format_func=lambda k: MODEL_FILES[k] +
                                 (" (best on validation)" if k == art["best"] else ""))
        show_lime = c2.checkbox("Explain with LIME", value=True)
        cross_ref = c2.checkbox("Cross-reference fact-checkers", value=True)
        submitted = st.form_submit_button("Check claim", type="primary")

    if submitted:
        run_check(mode, claim_in, url_in, model_key, show_lime, cross_ref)

# ---------------------------------------------------------------------------
# Search claims (FR6)
# ---------------------------------------------------------------------------
with tab_search:
    ds = load_dataset(_mtime(DATA_DIR / "claims_clean.csv"))
    st.write("Search claims already fact-checked by Dubawa and Africa Check, "
             "and claims checked through this app.")
    q = st.text_input("Keyword", placeholder="e.g. naira, INEC, vaccine")
    include_app = st.checkbox("Include claims checked through this app")
    if ds.empty:
        st.info("No dataset loaded yet.")
    else:
        f1, f2, f3 = st.columns(3)
        cats = f1.multiselect("Topic", sorted(ds["category"].unique()))
        srcs = f2.multiselect("Source", sorted(ds["source"].unique()))
        labs = f3.multiselect("Label", ["Real", "Fake"])
        res = ds.copy()
        if q:
            res = res[res["claim_text"].str.contains(q, case=False, regex=False)]
        if cats:
            res = res[res["category"].isin(cats)]
        if srcs:
            res = res[res["source"].isin(srcs)]
        if labs:
            res = res[res["label"].isin(labs)]
        st.caption(f"{len(res)} fact-checked claims")
        st.dataframe(res[["claim_text", "label", "verdict", "source", "category", "date", "url"]]
                     .head(300), hide_index=True, use_container_width=True,
                     column_config={"url": st.column_config.LinkColumn("url")})
    if include_app:
        lg = read_log()
        if not lg.empty:
            if q:
                lg = lg[lg["submitted_text"].str.contains(q, case=False, regex=False)]
            st.caption(f"{len(lg)} claims checked through the app")
            st.dataframe(lg[["created_at", "submitted_text", "label", "score_p_real",
                             "category", "model"]].head(300),
                         hide_index=True, use_container_width=True)
        else:
            st.caption("No claims have been checked through the app yet.")

# ---------------------------------------------------------------------------
# About
# ---------------------------------------------------------------------------
with tab_about:
    st.markdown(f"""
This prototype gives a fast first-line credibility assessment of claims circulating in
Nigerian online media, in English and Nigerian Pidgin. It is a **decision-support tool**
for the public and for fact-checkers - not an automated content-removal system.

**How it works:** the claim is cleaned (URLs, emojis and personal data removed,
Nigeria-adapted stopwords), its language is identified, it is turned into TF-IDF
features and scored by the selected classifier. LIME then shows which words drove
the result.

**Training data:** published verdicts from Dubawa and Africa Check (via their archives
and the Google Fact Check Tools API), harmonised to Real (True) and Fake (False,
Misleading). Unverifiable claims are excluded. All credit for the verdicts belongs to
those organisations.

**Current models:** trained {meta.get('trained_at', 'n/a')} on {meta.get('n_train', '?')}
training claims ({meta.get('n_val', '?')} validation, {meta.get('n_test', '?')} test).

**Limitations:** the model judges wording, not facts. New topics it has not seen, satire,
and very short claims can be misjudged. No personal data is stored; submitted text is
scrubbed of emails, phone numbers and handles before logging.
""")

# ---------------------------------------------------------------------------
# Admin (FR10, FR11, NFR8, NFR10)
# ---------------------------------------------------------------------------
with tab_admin:
    admin_pw = get_secret("ADMIN_PASSWORD")
    if not admin_pw:
        st.info("Admin is disabled. Set ADMIN_PASSWORD in the app secrets to enable it.")
    else:
        if not st.session_state.get("admin_ok"):
            pw = st.text_input("Admin password", type="password")
            if pw and hmac.compare_digest(pw, admin_pw):
                st.session_state["admin_ok"] = True
                st.rerun()
            elif pw:
                st.error("Wrong password.")
        if st.session_state.get("admin_ok"):
            st.subheader("Model performance")
            ms = MODEL_DIR / "metrics_summary.csv"
            if ms.exists():
                st.dataframe(pd.read_csv(ms), hide_index=True, use_container_width=True)
                st.download_button("Download metrics summary", ms.read_bytes(),
                                   "metrics_summary.csv", "text/csv")
            figdir = MODEL_DIR / "figures"
            if figdir.exists():
                for img in sorted(figdir.glob("*.png")):
                    st.image(str(img), use_container_width=True)

            st.subheader("Disaggregated performance (bias audit)")
            dg = MODEL_DIR / "disaggregated_report.csv"
            if dg.exists():
                d = pd.read_csv(dg, keep_default_na=False)
                a1, a2, a3 = st.columns(3)
                gcol = a1.selectbox("Group by", sorted(d["group_column"].unique()))
                split = a2.selectbox("Split", ["test", "val"])
                mdl = a3.selectbox("Model", sorted(d["model"].unique()))
                view = d[(d["group_column"] == gcol) & (d["split"] == split) & (d["model"] == mdl)]
                st.dataframe(view.drop(columns=["group_column", "split", "model"]),
                             hide_index=True, use_container_width=True)
                st.caption("false_positive_rate = genuine claims wrongly flagged as Fake. "
                           "Groups with fewer than 10 claims are flagged as small samples.")
                st.download_button("Download full disaggregated report", dg.read_bytes(),
                                   "disaggregated_report.csv", "text/csv")

            st.subheader("Classification log")
            lg = read_log()
            if lg.empty:
                st.caption("No classifications logged yet.")
            else:
                gcol2 = st.selectbox("Summarise log by",
                                     ["language_tag", "category", "region", "party_ref",
                                      "religion_ref", "ethnic_ref", "model", "input_type"])
                summ = lg.groupby([gcol2, "label"]).size().unstack(fill_value=0)
                summ["mean_score_p_real"] = lg.groupby(gcol2)["score_p_real"].mean().round(3)
                st.dataframe(summ, use_container_width=True)
                st.dataframe(lg.head(200), hide_index=True, use_container_width=True)
                st.download_button("Download log (CSV)", lg.to_csv(index=False).encode(),
                                   "classification_log.csv", "text/csv")

            st.subheader("Refresh data and retrain")
            st.caption("On Streamlit Community Cloud, files written here are lost when the app "
                       "restarts. For a permanent update, retrain in Colab and push the new "
                       "models/ and data/ folders to GitHub.")
            raw_files = sorted(RAW_DIR.glob("*.csv")) if RAW_DIR.exists() else []
            if raw_files:
                st.dataframe(pd.DataFrame([{"file": f.name,
                                            "rows": max(0, sum(1 for _ in open(f, encoding="utf-8")) - 1)}
                                           for f in raw_files]),
                             hide_index=True, use_container_width=True)
            up = st.file_uploader("Add newly labelled claims (CSV with claim_text, verdict, "
                                  "source columns)", type="csv")
            if up is not None:
                try:
                    new = pd.read_csv(up, dtype=str, keep_default_na=False)
                    missing = {"claim_text", "verdict"} - set(new.columns)
                    if missing:
                        st.error(f"Missing columns: {missing}")
                    else:
                        RAW_DIR.mkdir(parents=True, exist_ok=True)
                        name = f"uploaded_{datetime.now():%Y%m%d_%H%M%S}.csv"
                        new.to_csv(RAW_DIR / name, index=False)
                        st.success(f"Saved {len(new)} rows as data/raw/{name}")
                except Exception as e:
                    st.error(f"Could not read file: {e}")

            b1, b2 = st.columns(2)
            api_key = get_secret("GOOGLE_FACTCHECK_API_KEY")
            if b1.button("Fetch new claims from the API", disabled=not api_key):
                from fetch_factcheck_api import run as fetch_run
                buf = io.StringIO()
                with st.spinner("Fetching..."), contextlib.redirect_stdout(buf):
                    try:
                        fetch_run(api_key, out_path=str(RAW_DIR / "api_claims.csv"))
                    except Exception as e:
                        print(f"Error: {e}")
                st.code(buf.getvalue()[-4000:])
            if b2.button("Rebuild dataset and retrain", type="primary"):
                import preprocess
                import train_model
                buf = io.StringIO()
                with st.spinner("Preprocessing and training - this can take a few minutes..."), \
                        contextlib.redirect_stdout(buf):
                    try:
                        preprocess.run(raw_dir=str(RAW_DIR), outdir=str(DATA_DIR))
                        train_model.run(str(DATA_DIR), str(MODEL_DIR))
                        ok = True
                    except Exception as e:
                        print(f"Error: {e}")
                        ok = False
                st.code(buf.getvalue()[-6000:])
                if ok:
                    st.cache_resource.clear()
                    st.cache_data.clear()
                    st.success("Retrained. The app now uses the new models.")

In [ ]:
%%writefile tests/test_preprocess.py
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from preprocess import (clean_text, detect_language, harmonize_label,  # noqa: E402
                        remove_duplicates, scrub_pii, tag_groups)


# ---- label harmonisation (3.4.2.3) ----
def test_harmonize_real():
    for v in ["True", "VERIFIED", "correct", "Mostly True", "mostly-true", "Mostly Correct"]:
        assert harmonize_label(v) == "Real"


def test_harmonize_fake():
    for v in ["False", "misleading", "EXAGGERATED", "mostly_false", "Fake", "Incorrect",
              "False!", "Partly False"]:
        assert harmonize_label(v) == "Fake"


def test_harmonize_sentence_ratings():
    assert harmonize_label("False. The photo is from 2019 in Ghana") == "Fake"
    assert harmonize_label("Misleading: the figure is outdated") == "Fake"
    assert harmonize_label("True - INEC confirmed this") == "Real"


def test_harmonize_drops_unverifiable():
    for v in ["Unverifiable", "Unclear", "", "Unproven", "Half true", "Satire", None]:
        assert harmonize_label(v) is None


def test_harmonize_does_not_confuse_prefixes():
    assert harmonize_label("Truecaller app") is None  # 'true' must be a whole word


# ---- cleaning (3.4.2.4) ----
def test_clean_preserves_numbers_in_words():
    out = clean_text("covid19 and 5g rollout")
    assert "covid19" in out.split() and "5g" in out.split()


def test_clean_drops_standalone_digits():
    assert "2023" not in clean_text("the year 2023 was")


def test_clean_removes_html_and_emojis():
    out = clean_text("<p>Fake news \U0001F621 here</p>")
    assert "<" not in out and "\U0001F621" not in out
    assert "pfake" not in out and "herep" not in out


def test_clean_removes_urls_and_personal_data():
    out = clean_text("Call 08031234567 or mail john@example.com see https://x.co/a @handle")
    for bad in ["0803", "john", "example", "https", "handle"]:
        assert bad not in out


def test_clean_keeps_negation():
    assert "not" in clean_text("INEC did not announce results", lang="en").split()


def test_clean_keeps_pidgin_content_words():
    out = clean_text("Wahala dey for Lagos, sabi pipu")
    assert "wahala" in out
    assert "sabi" in out


def test_stemming_only_for_english():
    assert "elect" in clean_text("elections", lang="en")
    assert "elections" in clean_text("elections", lang="pcm")


def test_scrub_pii():
    s = scrub_pii("Reach me on +2348031234567 or a@b.com @me")
    assert "[phone]" in s and "[email]" in s and "[user]" in s


# ---- language identification ----
def test_detect_language():
    assert detect_language("Wahala dey for Lagos") == "pcm"
    assert detect_language("Fuel subsidy don finish for Nigeria") == "pcm"
    assert detect_language("Hot water fit cure malaria") == "pcm"
    assert detect_language("The president signed the budget yesterday") == "en"


# ---- group tags ----
def test_tag_groups():
    g = tag_groups("APC governor in Kano says vaccines cause infertility")
    assert g["region"] == "North-West"
    assert g["party_ref"] == "APC"
    assert g["category"] in {"politics", "health"}
    assert tag_groups("Nothing specific here")["region"] == "National/Unspecified"


# ---- duplicates and conflicting verdicts ----
def test_dedup_exact_and_conflict():
    df = pd.DataFrame({
        "claim_text": ["INEC bans BVAS", "inec bans bvas!", "Naira is gone", "naira is GONE",
                       "Lagos floods"],
        "label": ["Fake", "Fake", "Fake", "Real", "Real"],
        "category": ["politics", "politics", "other", "other", "other"],
    })
    log = {}
    out = remove_duplicates(df, log)
    assert log["exact_duplicates_removed"] == 1
    assert log["conflicting_verdicts_removed"] == 2
    assert len(out) == 2


def test_near_duplicates_removed():
    df = pd.DataFrame({
        "claim_text": ["The CBN has recalled all old naira notes permanently today",
                       "The CBN has recalled all old naira notes permanently today now",
                       "Drinking hot water cures malaria"],
        "label": ["Fake", "Fake", "Fake"],
        "category": ["other", "other", "other"],
    })
    log = {}
    out = remove_duplicates(df, log)
    assert log["near_duplicates_removed"] == 1
    assert len(out) == 2

In [ ]:
%%writefile tests/test_scrapers.py
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from fetch_factcheck_api import _rows_from_response  # noqa: E402
from scrape_factcheckers import (parse_africacheck_article,  # noqa: E402
                                 parse_africacheck_listing, parse_dubawa_post)


def test_parse_dubawa_post():
    content = ("<p><strong>Claim:</strong> A Facebook post claims Lagos has banned okada.</p>"
               "<p><strong>Verdict:</strong> False! The ban covers only some LGAs.</p>"
               "<h3>Full Text:</h3><p>...</p>")
    claim, verdict, title, _ = parse_dubawa_post("False! Lagos did not ban okada", content)
    assert claim.startswith("A Facebook post claims Lagos")
    assert verdict == "False"


def test_parse_dubawa_title_fallback():
    claim, verdict, _, _ = parse_dubawa_post("Misleading! Viral video of flood", "<p>Text</p>")
    assert verdict == "Misleading"


def test_parse_africacheck_listing():
    html = ('<a href="/fact-checks/fact-checks/no-nigeria-did-not-ban-x">a</a>'
            '<a href="https://africacheck.org/fact-checks/reports/some-report">b</a>'
            '<a href="/about-us">c</a>')
    links = parse_africacheck_listing(html)
    assert "https://africacheck.org/fact-checks/fact-checks/no-nigeria-did-not-ban-x" in links
    assert len(links) == 2


def test_parse_africacheck_jsonld():
    html = ('<html><h1>Title</h1><script type="application/ld+json">'
            '{"@graph":[{"@type":"ClaimReview","claimReviewed":"Nigeria has 300m people",'
            '"reviewRating":{"alternateName":"Incorrect"}}]}</script></html>')
    claim, verdict, title, _ = parse_africacheck_article(html)
    assert claim == "Nigeria has 300m people" and verdict == "Incorrect" and title == "Title"


def test_parse_africacheck_rating_image():
    html = ('<html><h1>T</h1><p>IN SHORT: A viral claim says X.</p>'
            '<img src="/files/Rating_Misleading.png" alt=""></html>')
    claim, verdict, _, _ = parse_africacheck_article(html)
    assert claim == "A viral claim says X." and verdict == "Misleading"


def test_api_publisher_filter():
    data = {"claims": [{"text": "Claim A", "claimReview": [
        {"publisher": {"name": "Dubawa", "site": "dubawa.org"}, "textualRating": "False",
         "url": "https://dubawa.org/a"},
        {"publisher": {"name": "Other", "site": "other.com"}, "textualRating": "True",
         "url": "https://other.com/a"}]}]}
    rows = _rows_from_response(data, ["dubawa.org", "africacheck.org"])
    assert len(rows) == 1 and rows[0]["source"] == "Dubawa"

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#2a6f97"
font = "sans serif"

[server]
maxUploadSize = 20

In [ ]:
%%writefile .gitignore
__pycache__/
*.pyc
.ipynb_checkpoints/
logs/
.streamlit/secrets.toml
.venv/
venv/
*.zip

In [ ]:
%%writefile data/raw/sample_claims.csv
claim_text,verdict,source,language_tag
The Nigerian government has removed fuel subsidy in 2023,True,Dubawa,en
INEC announced results before voting started,False,Africa Check,en
5G network causes COVID-19,False,Dubawa,en
Nigeria's population is over 200 million,True,Africa Check,en
Tinubu was sworn in as president on May 29 2023,True,Dubawa,en
Naira has been replaced with a new currency,False,Africa Check,en
Boko Haram has been completely defeated,False,Dubawa,en
Nigeria gained independence in 1960,True,Africa Check,en
Drinking hot water cures malaria,False,Dubawa,en
Lagos is the commercial capital of Nigeria,True,Africa Check,en
A new law bans all motorcycles nationwide,False,Dubawa,en
Nigeria has one of Africa's largest economies,True,Africa Check,en
BVAS machines were used in the 2023 elections,True,Dubawa,en
COVID-19 vaccines contain microchips,False,Africa Check,en
The CBN has recalled all old naira notes permanently,False,Dubawa,en
Fuel subsidy don finish for Nigeria,True,Dubawa,pcm
Dem tok say INEC announce result before election,False,Dubawa,pcm
"Wahala dey for Lagos, flood don carry everybody",False,Dubawa,pcm
Naija population pass 200 million,True,Dubawa,pcm
Tinubu don become president for May 29,True,Dubawa,pcm
Dem say naira don change to new currency,False,Dubawa,pcm
Boko Haram don finish completely for Nigeria,False,Dubawa,pcm
Naija gain independence for 1960,True,Dubawa,pcm
Hot water fit cure malaria,False,Dubawa,pcm
Lagos na commercial capital of Naija,True,Dubawa,pcm
New law ban all okada for whole country,False,Dubawa,pcm
Naija get one of the biggest economy for Africa,True,Dubawa,pcm
BVAS machine dem use for 2023 election,True,Dubawa,pcm
COVID vaccine get microchip inside,False,Dubawa,pcm
CBN don recall all old naira note,False,Dubawa,pcm
Aliens landed in Abuja last week,Unverifiable,Dubawa,en
Dem say alien land for Abuja,Unverifiable,Dubawa,pcm

## Step 3 - Install libraries
If Colab shows a **Restart session** button after this, click it, then run Step 1 again and continue from Step 4 (you do not need to re-run Step 2).

In [ ]:
!pip install -q -r requirements.txt
import sklearn, imblearn
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__)

## Step 4 - Collect the data
The sample file (32 claims) only proves the pipeline runs. For the real study (~3,000 claims) use **4A** and **4B** (and **4C** for anything collected by hand). Once real files exist in `data/raw/`, the sample is excluded automatically.

### 4A - Google Fact Check Tools API (Dubawa + Africa Check only)
Get a free key: Google Cloud Console -> APIs & Services -> enable **Fact Check Tools API** -> Credentials -> **Create API key**.
Store it in Colab: key icon in the left sidebar -> **Add new secret** -> name `GOOGLE_FACTCHECK_API_KEY` -> paste the key -> enable notebook access.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_FACTCHECK_API_KEY"] = userdata.get("GOOGLE_FACTCHECK_API_KEY")
except Exception:
    os.environ["GOOGLE_FACTCHECK_API_KEY"] = ""  # or paste the key here as a string

if os.environ["GOOGLE_FACTCHECK_API_KEY"]:
    !python fetch_factcheck_api.py --max-pages 5
else:
    print("No API key set - skipping 4A.")

### 4B - Scrape the Dubawa and Africa Check archives
Uses Requests + BeautifulSoup with a polite delay, so it is slow (roughly 1-2 seconds per article). Increase `--max-pages` to collect more.
Claims whose verdict cannot be read reliably go to `data/raw/needs_review_*.csv` instead of being guessed (see 4D).

In [ ]:
!python scrape_factcheckers.py --site dubawa --max-pages 20
!python scrape_factcheckers.py --site africacheck --max-pages 20

### 4C - Upload your own labelled CSV (optional)
Columns needed: `claim_text`, `verdict` (the fact-checker's own rating word, e.g. False / Misleading / True), `source`. Optional: `url`, `date`, `category`, `language_tag`.

In [ ]:
from google.colab import files
import shutil
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f"data/raw/{name}")
    print("Added data/raw/" + name)

### 4D - Fill in the claims that need review (optional)
Download the review files, open each article URL, type the published verdict into the empty `verdict` column, then upload the file back with the **same name**. Only rows with a verdict filled in are used.

In [ ]:
import glob
from google.colab import files
for f in glob.glob("data/raw/needs_review_*.csv"):
    files.download(f)

### Check what was collected

In [ ]:
import glob
import pandas as pd
for f in sorted(glob.glob("data/raw/*.csv")):
    df = pd.read_csv(f, dtype=str, keep_default_na=False)
    print(f"{f:45s} {len(df):6d} rows")

## Step 5 - Preprocess
Label harmonisation -> language identification -> topic/region/party/religion/ethnic tags -> exact + near-duplicate and conflicting-verdict removal -> cleaning -> stratified 80/10/10 split.
The log (`data/preprocess_log.json`) gives the counts Chapter Four needs (rows dropped, duplicates removed, conflicts, class and language balance).

In [ ]:
!python preprocess.py
import json
print(json.dumps(json.load(open("data/preprocess_log.json")), indent=2))

## Step 6 - Train the three classifiers
TF-IDF is fitted on the training split only. Each classifier is tuned with stratified k-fold cross-validation; SMOTE runs inside each training fold only. The best model is chosen on validation macro-F1 and its test score is the unbiased estimate.

In [ ]:
!python train_model.py

## Step 7 - Results for Chapter Four

In [ ]:
import pandas as pd
summary = pd.read_csv("models/metrics_summary.csv")
summary

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("models/figures/*.png")):
    print(f)
    display(Image(f))

In [ ]:
# Bias audit: performance by group on the test set (false_positive_rate = genuine claims flagged as Fake)
d = pd.read_csv("models/disaggregated_report.csv", keep_default_na=False)
for col in ["language_tag", "category", "source", "region", "party_ref", "religion_ref", "ethnic_ref"]:
    print(f"\n=== {col} ===")
    display(d[(d.group_column == col) & (d.split == "test")]
            .drop(columns=["group_column", "split"]).round(3))

## Step 8 - Run the tests

In [ ]:
!python -m pytest -q

## Step 9 - Try some claims in the notebook

In [ ]:
import joblib
from preprocess import clean_text, detect_language

vectorizer = joblib.load("models/vectorizer.pkl")
names = {"naive_bayes": "Naive Bayes", "logistic_regression": "Logistic Regression",
         "random_forest": "Random Forest"}
models = {k: joblib.load(f"models/{k}.pkl") for k in names}

claims = [
    "The Nigerian government has removed fuel subsidy",
    "Fuel subsidy don finish for Naija",
    "5G network causes COVID-19",
    "5G network dey cause COVID",
    "INEC has postponed the governorship election in Rivers State",
]
for c in claims:
    lang = detect_language(c)
    vec = vectorizer.transform([clean_text(c, lang=lang)])
    print(f"\n{c!r}  [{lang}]")
    for k, m in models.items():
        p_real = m.predict_proba(vec)[0][list(m.classes_).index("Real")]
        print(f"  {names[k]:20s} -> {'Real' if p_real >= 0.5 else 'Fake':4s}  credibility {p_real:.2f}")

## Step 10 - Preview the app inside Colab (temporary)
This link only works while this notebook is open. The permanent link comes from Step 12.
If the page stays blank, use the ngrok cell below instead.

In [ ]:
import subprocess, time
app_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501",
                             "--server.headless", "true"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
from google.colab.output import eval_js
print("Open the app:", eval_js("google.colab.kernel.proxyPort(8501)"))

In [ ]:
# Alternative preview via ngrok (free account needed: https://dashboard.ngrok.com -> Your Authtoken)
!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("PASTE_YOUR_NGROK_TOKEN")
ngrok.kill()
print("Public URL:", ngrok.connect(8501, "http"))

In [ ]:
# Stop the preview
try:
    app_proc.terminate()
except NameError:
    pass
try:
    from pyngrok import ngrok
    ngrok.kill()
except Exception:
    pass

## Step 11 - Download the deploy-ready project
This zip is the GitHub repository: code, cleaned dataset, trained models and figures. It also goes in the project appendix.

In [ ]:
import os, zipfile
from google.colab import files
SKIP = {"logs", "__pycache__", ".pytest_cache", ".ipynb_checkpoints"}
out = "/content/fake-news-detector-ng.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fnames in os.walk(PROJECT):
        dirs[:] = [d for d in dirs if d not in SKIP]
        for fn in fnames:
            full = os.path.join(root, fn)
            z.write(full, os.path.join("fake-news-detector-ng", os.path.relpath(full, PROJECT)))
files.download(out)

## Step 12 - Deploy (permanent public link)
1. Unzip, create a GitHub repository, and upload all the files (keep the folder structure: `models/`, `data/`, `.streamlit/`).
2. Go to share.streamlit.io -> **Create app** -> pick the repository, branch `main`, main file `app.py`.
3. Under **Advanced settings** choose Python **3.12** and add secrets:
   ```
   GOOGLE_FACTCHECK_API_KEY = "your-key"
   ADMIN_PASSWORD = "choose-a-password"
   ```
4. Deploy. After any retraining here, upload the new `models/` and `data/` folders to GitHub and the app updates itself.